# EGATUR / INE: gasto turistico en Espana

**Que responde este cuaderno**

1. Como evoluciona el gasto turistico total mes a mes y cuanto varia de un ano a otro.
2. Que paises de residencia aportan mas gasto y con que cuota.
3. Que comunidades autonomas concentran ese gasto.
4. Por que via llegan los turistas que mas gastan.
5. Que motivos de viaje mueven el gasto, en la ultima anualidad disponible.

**Datos**: `data/processed/egatur_master_long.csv` (2.881 filas en formato largo: una fila por tabla, periodo, dimension e indicador). Se descarga por URL al ejecutar: no hace falta subir nada.

**Como correrlo**: Runtime > Run all. Solo pandas y matplotlib.

**Unidades**: el CSV trae el valor tal como lo publica el INE, sin columna de escala. Por eso se trabaja con cuotas y variaciones, no con absolutos convertidos.

**Ojo con los periodos**: las tablas de pais, CCAA y via de acceso son mensuales; la de motivo del viaje es anual. El cuaderno separa ambos casos.

In [ ]:
import io, pathlib, urllib.request
import pandas as pd
import matplotlib.pyplot as plt

BASE = 'https://raw.githubusercontent.com/Nicolenki7/ETL_Egatur_INE_Esp/main/'
RUTA = 'data/processed/egatur_master_long.csv'
pathlib.Path('outputs').mkdir(exist_ok=True)

with urllib.request.urlopen(BASE + RUTA) as r:
    d = pd.read_csv(io.BytesIO(r.read()), dtype={'periodo_parsed': str})

d['es_mensual'] = d['periodo_parsed'].str.match(r'^\d{4}-\d{2}$')
d['fecha'] = pd.to_datetime(d.loc[d['es_mensual'], 'periodo_parsed'], format='%Y-%m')
mensual = d[d['es_mensual']].copy()

print('filas:', len(d), '| mensuales:', len(mensual), '| anuales:', len(d) - len(mensual))
print('rango mensual:', mensual['periodo_parsed'].min(), 'a', mensual['periodo_parsed'].max())
print()
print(d.groupby('table_name').agg(filas=('valor', 'size'), dimensiones=('dimension_value', 'nunique')).to_string())

## 1. Gasto total por mes

La tabla 10838 incluye, ademas de cada pais, una fila `Total` por periodo. Esa es la serie agregada: la usamos tal cual, sin sumar los paises (eso duplicaria el total).

In [ ]:
total = (mensual[(mensual['table_id'] == 10838) & (mensual['dimension_value'] == 'Total')]
         .drop_duplicates('periodo_parsed')
         .sort_values('fecha'))

ultimo = total.iloc[-1]
suma_ult = total.tail(12)['valor'].sum()
suma_prev = total.iloc[-24:-12]['valor'].sum()
var = (suma_ult / suma_prev - 1) * 100

print(f'RESPUESTA: en {ultimo.periodo_parsed} el gasto total fue {ultimo.valor}; los ultimos 12 meses suman {var:.1f} % mas que los 12 anteriores.')

ax = total.tail(60).set_index('fecha')['valor'].plot(figsize=(9, 3.5))
ax.set_ylabel('gasto total (unidad del INE)')
ax.set_title('Gasto turistico total por mes (ultimos 5 anos)')
plt.tight_layout()
plt.savefig('outputs/gasto_mensual.png', dpi=150)
plt.show()

## 2. Que paises sostienen el gasto

Excluimos la fila `Total` para no contar dos veces el agregado y comparamos el ultimo ano movil completo.

In [ ]:
inicio = total['periodo_parsed'].iloc[-12]
corte = total['periodo_parsed'].iloc[-1]

def recorte(table_id):
    g = mensual[(mensual['table_id'] == table_id) & (mensual['dimension_value'] != 'Total')]
    return g[(g['periodo_parsed'] >= inicio) & (g['periodo_parsed'] <= corte)]

paises = recorte(10838)
por_pais = paises.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota = (por_pais / por_pais.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_pais.round(1), 'cuota_pct': cuota}).head(8).to_string())

lista = ', '.join(cuota.head(3).index)
print()
print(f'RESPUESTA: {lista} aportan juntos el {cuota.head(3).sum():.1f} % del gasto del ultimo ano.')

ax = cuota.head(6).sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% del gasto (ultimos 12 meses)')
ax.set_title('Cuota por pais de residencia')
plt.tight_layout()
plt.savefig('outputs/cuota_pais.png', dpi=150)
plt.show()

## 3. Destino: comunidades autonomas

Misma logica con la tabla 10839 (CCAA destino).

In [ ]:
por_ccaa = recorte(10839).groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota_ccaa = (por_ccaa / por_ccaa.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_ccaa.round(1), 'cuota_pct': cuota_ccaa}).head(8).to_string())

print()
print(f'RESPUESTA: {cuota_ccaa.index[0]} lidera con el {cuota_ccaa.iloc[0]} % del gasto; las tres primeras suman el {cuota_ccaa.head(3).sum():.1f} %.')

## 4. Via de acceso

La tabla 10828 reparte el gasto segun como entra el turista (avion, carretera, etc.).

In [ ]:
por_via = recorte(10828).groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota_via = (por_via / por_via.sum() * 100).round(1)
print(pd.DataFrame({'gasto': por_via.round(1), 'cuota_pct': cuota_via}).to_string())

print()
print(f'RESPUESTA: {cuota_via.index[0]} es la via dominante con el {cuota_via.iloc[0]} % del gasto.')

ax = cuota_via.sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% del gasto')
ax.set_title('Gasto por via de acceso (ultimos 12 meses)')
plt.tight_layout()
plt.savefig('outputs/cuota_via.png', dpi=150)
plt.show()

## 5. Motivo del viaje (anual)

La tabla 23995 es anual: se toma la ultima anualidad disponible, no se mezcla con la serie mensual.

In [ ]:
motivos = d[(d['table_id'] == 23995) & (d['dimension_value'] != 'Total')]
anio = motivos['periodo_parsed'].max()
m = motivos[motivos['periodo_parsed'] == anio]

por_motivo = m.groupby('dimension_value')['valor'].sum().sort_values(ascending=False)
cuota_motivo = (por_motivo / por_motivo.sum() * 100).round(1)
print('anualidad:', anio)
print(pd.DataFrame({'gasto': por_motivo.round(1), 'cuota_pct': cuota_motivo}).head(8).to_string())

print()
print(f'RESPUESTA: en {anio} el motivo con mas gasto es {cuota_motivo.index[0]}, con el {cuota_motivo.iloc[0]} %.')

ax = cuota_motivo.head(8).sort_values().plot(kind='barh', figsize=(7, 3.5))
ax.set_xlabel('% del gasto anual')
ax.set_title('Gasto por motivo del viaje (' + anio + ')')
plt.tight_layout()
plt.savefig('outputs/cuota_motivo.png', dpi=150)
plt.show()

## Limites

- El CSV es el maestro del proyecto: no consulta el INE en vivo.
- `tipo_dato` es 'Dato base' en todas las filas; no hay series corregidas ni desestacionalizadas.
- La unidad no viene en el archivo: se leen cuotas y variaciones, no absolutos convertidos.
- La tabla 13938 es una unica serie de gasto total, no un desglose por partidas: por eso no se usa como categoria.
- Algunos 'paises' son agrupaciones del INE (por ejemplo Paises Nordicos).
- Coincidencia de nombres exacta: si el INE renombra una categoria, aparece como valor nuevo.